# 05 — Feature Engineering

## Objective
Create useful features for predicting next-week HAB severity
and remove highly redundant predictors.

This notebook will:
1. Review existing features.
2. Engineer HAB history, trend and duration features.
3. Engineer seasonal features.
4. Engineer selected environmental features.
5. Add the regional features to the site-week table.
6. Identify and remove highly redundant predictors.
7. Define the feature sets for modelling, then validate and export the datasets.

### Updated for the October tables
Notebook 03 now builds a **regional** table (595 weeks, 2015–2026, focus-coast label, 5k / 100k cutoffs) and a
**site-week** table (21,718 rows). The HAB lag and change features below are now computed on the **last observed**
class and count, which are carried forward through unsampled weeks, so they are never missing after the first sample.


## 1. Load the Weekly Modeling Dataset


In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Data folder: our Google Drive folder in Colab, otherwise the repo's data/ folder
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/biointerphase_data"
except ImportError:
    DATA = os.path.abspath(os.path.join("..", "data"))

PROCESSED = os.path.join(DATA, "processed")
data_path = os.path.join(PROCESSED, "model_table_weekly.parquet")
site_path = os.path.join(PROCESSED, "site_week_table.parquet")

# Load the datasets
df = pd.read_parquet(data_path)
site_df = pd.read_parquet(site_path)

# Create working copies for feature engineering
df_fe = df.copy()
site_fe = site_df.copy()

# Display dataset information
print("Datasets loaded successfully!")
print("Regional table:", df.shape[0], "rows x", df.shape[1], "columns")
print("Site-week table:", site_df.shape[0], "rows x", site_df.shape[1], "columns")

# Preview the first five rows
display(df.head())

Datasets loaded successfully!
Regional table: 595 rows x 101 columns
Site-week table: 21718 rows x 37 columns


,week_start,target_week,target_year,bloom_season,split,y_class,y_binary,y_log_cells,y_n_samples,y_n_sites,label_effort_ok,y_class_10k,y_class_box,y_class_mote_25km,hab_class,n_habsos_samples,n_habsos_sites,n_habsos_days,hab_pos_share,hab_bloom_share,hab_detect_share,hab_log_cells_p90,last_class,last_log_cells,weeks_since_obs,class_max_4wk,log_cells_trend,nutr_bay_chla_corr_ugL_anom,nutr_bay_cond_umho_anom,nutr_bay_do_mgL_anom,nutr_bay_nox_mgL_anom,nutr_bay_po4_mgL_anom,nutr_bay_tkn_mgL_anom,nutr_bay_tn_mgL_anom,nutr_bay_tp_mgL_anom,nutr_bay_water_temp_c_anom,nutr_fresh_chla_corr_ugL_anom,nutr_fresh_cond_umho_anom,nutr_fresh_do_mgL_anom,nutr_fresh_nh3_mgL_anom,nutr_fresh_nox_mgL_anom,nutr_fresh_po4_mgL_anom,nutr_fresh_tkn_mgL_anom,nutr_fresh_tn_mgL_anom,nutr_fresh_tp_mgL_anom,nutr_fresh_water_temp_c_anom,nutr_tidal_chla_corr_ugL_anom,nutr_tidal_cond_umho_anom,nutr_tidal_do_mgL_anom,nutr_tidal_nh3_mgL_anom,nutr_tidal_nox_mgL_anom,nutr_tidal_po4_mgL_anom,nutr_tidal_tkn_mgL_anom,nutr_tidal_tn_mgL_anom,nutr_tidal_tp_mgL_anom,nutr_tidal_water_temp_c_anom,nutr_bay_n_stations,nutr_fresh_n_stations,nutr_tidal_n_stations,nutr_age_weeks,rain_total_mm,rain_days,air_tmax_c,air_tmin_c,air_tmean_c,air_trange_c,ksrq_pressure_hpa,ksrq_air_temp_c,ksrq_dew_point_c,ksrq_wind_speed,ksrq_wind_u,ksrq_wind_v,has_ksrq,uwnd_box,uwnd_offshore,uwnd_onshore_days,uwnd_westerly_mean,uwnd_easterly_mean,mote_temp_mean_c,mote_temp_max_c,has_mote,chl_box_med,chl_coast20_med,chl_focus_0_10_med,chl_focus_10_40_med,chl_focus_0_40_med,chl_box_n_valid,chl_coast20_anom,chl_focus_0_10_anom,chl_focus_0_40_anom,chl_focus_frac_gt3,chl_focus_frac_gt10,chl_qc_box_cover,rain_2wk_mm,rain_4wk_mm,rain_8wk_mm,uwnd_box_4wk,chl_focus_0_40_anom_4wk,chl_focus_0_10_med_4wk,chl_focus_10_40_med_4wk,chl_focus_0_40_med_chg
0,2015-01-05,2015-01-12,2015,2014,train,0.0,0.0,0.0,29.0,23.0,1,0.0,0.0,0.0,0.0,35.0,29.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,NaN,0.147409,-0.003831,-0.048837,-0.040026,-0.090067,0.190594,0.185287,-0.158867,3.501512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004439,-0.089638,NaN,0.002338,0.104062,0.169605,0.168584,-0.010979,3.571379,30.0,0.0,10.0,0.0,0.000,0.0,27.222222,4.444444,15.436508,10.555556,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-12,2015-01-19,2015,2014,train,0.0,0.0,0.0,42.0,36.0,1,0.0,0.0,0.0,0.0,29.0,23.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,NaN,0.147409,-0.003831,-0.048837,-0.040026,-0.090067,0.190594,0.185287,-0.158867,3.501512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004439,-0.089638,NaN,0.002338,0.104062,0.169605,0.168584,-0.010979,3.571379,0.0,0.0,0.0,1.0,8.636,1.0,25.555556,8.333333,17.301587,8.888889,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.636,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-19,2015-01-26,2015,2014,train,0.0,0.0,0.0,33.0,26.0,1,0.0,0.0,0.0,0.0,42.0,36.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.147409,-0.003831,-0.048837,-0.040026,-0.090067,0.190594,0.185287,-0.158867,3.501512,-0.004154,0.088848,-0.082732,-0.270426,0.140403,0.172478,0.014827,0.032515,0.068805,-0.030660,0.078249,0.143224,-0.010227,-0.211535,0.203963,0.065072,0.052314,0.053991,0.031591,0.897214,0.0,20.0,9.0,0.0,59.436,2.0,26.666667,7.777778,17.658730,11.507937,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.072,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-26,2015-02-02,2015,2014,train,0.0,0.0,0.0,71.0,65.0,1,0.0,0.0,0.0,0.0,33.0,26.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.147409,-0.003831,-0.048837,-0.040026,-0.090067,0.190594,0.185287,-0.158867,3.501512,-0.095957,-0.032097,-0.118392,0.011921,0.076042,0.196484,0.050216,0.022397,0.075716,0.928334,0.072324,-0.108068,-0.019858,-0.098245,-0.258815,0.187771,0.039312,0.005644,0.129043,0.571572,0.0,14.0,8.0,0.0,0.508,1.0,24.444444,6.111111,14.761905,11.904762,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,

#### Display all existing features

In [3]:

# Create a feature inventory
feature_inventory = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

# Start numbering from 1
feature_inventory.index = range(
    1, len(feature_inventory) + 1
)

print("EXISTING FEATURE INVENTORY")
print("-" * 40)

display(feature_inventory)

print("\nTotal columns:", len(df.columns))
print(
    "Duplicate column names:",
    df.columns.duplicated().sum()
)

EXISTING FEATURE INVENTORY
----------------------------------------


,Feature,Data Type
1,week_start,datetime64[us]
2,target_week,datetime64[us]
3,target_year,int32
4,bloom_season,int32
5,split,str
...,...,...
97,uwnd_box_4wk,float64
98,chl_focus_0_40_anom_4wk,float64
99,chl_focus_0_10_med_4wk,float64
100,chl_focus_10_40_med_4wk,float64



Total columns: 101
Duplicate column names: 0


In [4]:

# Keywords commonly used in engineered feature names
engineering_keywords = [
    "lag",
    "last",
    "rolling",
    "anom",
    "2wk",
    "4wk",
    "8wk",
    "sin",
    "cos",
    "chg",
    "trend"
]

# Identify candidate engineered features
existing_engineered = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in engineering_keywords
    )
]

print("POTENTIALLY ENGINEERED FEATURES")
print("-" * 40)

for feature in existing_engineered:
    print("-", feature)

print(
    "\nNumber identified:",
    len(existing_engineered)
)

POTENTIALLY ENGINEERED FEATURES
----------------------------------------
- last_class
- last_log_cells
- weeks_since_obs
- class_max_4wk
- log_cells_trend
- nutr_bay_chla_corr_ugL_anom
- nutr_bay_cond_umho_anom
- nutr_bay_do_mgL_anom
- nutr_bay_nox_mgL_anom
- nutr_bay_po4_mgL_anom
- nutr_bay_tkn_mgL_anom
- nutr_bay_tn_mgL_anom
- nutr_bay_tp_mgL_anom
- nutr_bay_water_temp_c_anom
- nutr_fresh_chla_corr_ugL_anom
- nutr_fresh_cond_umho_anom
- nutr_fresh_do_mgL_anom
- nutr_fresh_nh3_mgL_anom
- nutr_fresh_nox_mgL_anom
- nutr_fresh_po4_mgL_anom
- nutr_fresh_tkn_mgL_anom
- nutr_fresh_tn_mgL_anom
- nutr_fresh_tp_mgL_anom
- nutr_fresh_water_temp_c_anom
- nutr_tidal_chla_corr_ugL_anom
- nutr_tidal_cond_umho_anom
- nutr_tidal_do_mgL_anom
- nutr_tidal_nh3_mgL_anom
- nutr_tidal_nox_mgL_anom
- nutr_tidal_po4_mgL_anom
- nutr_tidal_tkn_mgL_anom
- nutr_tidal_tn_mgL_anom
- nutr_tidal_tp_mgL_anom
- nutr_tidal_water_temp_c_anom
- chl_coast20_anom
- chl_focus_0_10_anom
- chl_focus_0_40_anom
- rain_2wk_mm
- 


## 1. HAB History and Trend Feature Engineering


Now we will be creating new features that capture the current intensity, historical severity, and recent changes in Harmful Algal Bloom (HAB) conditions.

The current-week class (`hab_class`) is missing in every week without a sample (18% of weeks). Lags and changes of it
would be missing too, so every feature here starts from the **carried-forward** columns made in notebook 03:
`last_class` (last observed class), `last_log_cells` (log10 of its max count) and `weeks_since_obs` (their age).

We engineer these candidate features:

1. **last_class_prev:** last observed class one week earlier (the carried version of `hab_class_lag1`).
2. **class_change_1wk:** `last_class` minus `last_class_prev` (−2 to 2).
3. **last_log_cells_chg1:** one-week change in the log10 cell count (the carried version of `hab_log_change_1wk`).
4. **class_run_weeks:** how many consecutive weeks the current class has lasted.
5. **weeks_since_pos / weeks_since_bloom:** weeks since a sampled week reached ≥ 5,000 / ≥ 100,000 cells/L.
6. **pos_weeks_12wk:** how many of the last 12 weeks were sampled at ≥ 5,000 cells/L.
7. **last_fwc_level:** FWC abundance category (0 background ≤ 1k, 1 very low, 2 low, 3 medium, 4 high > 1M) of the last observed count.

### Benefits

- Capture recent HAB history without gaps in unsampled weeks.
- Identify increasing or decreasing bloom intensity.
- **Duration:** blooms last for months, so how long the current state has lasted carries information that this week's class does not.

### Missing Values

Missing observations are preserved. No imputation is performed; the carried-forward columns come from notebook 03.

In [5]:
# HAB History and Trend Feature Engineering

# Ensure the date column is datetime and rows are in time order
df_fe["week_start"] = pd.to_datetime(df_fe["week_start"])
df_fe = df_fe.sort_values("week_start").reset_index(drop=True)

# Check for duplicate weeks
assert not df_fe.duplicated(subset=["week_start"]).any(), "Duplicate weeks found."

# Verify consecutive weekly observations
week_gaps = df_fe["week_start"].diff().dropna()

assert week_gaps.eq(
    pd.Timedelta(days=7)
).all(), "Non-consecutive weeks found. Review before creating lags."


def weeks_since(flag):
    """Weeks since `flag` was last True (0 = this week); NaN before the first True."""
    flag = flag.fillna(False).astype(bool)
    block = flag.cumsum()
    return flag.groupby(block).cumcount().where(block > 0).astype(float)


def run_length(series):
    """Number of consecutive rows (ending at each row) with the same value."""
    block = (series != series.shift()).cumsum()
    return (series.groupby(block).cumcount() + 1).where(series.notna()).astype(float)


# Features 1-3: lag and change on the carried-forward values
df_fe["last_class_prev"] = df_fe["last_class"].shift(1)
df_fe["class_change_1wk"] = df_fe["last_class"] - df_fe["last_class_prev"]
df_fe["last_log_cells_chg1"] = df_fe["last_log_cells"] - df_fe["last_log_cells"].shift(1)

# Features 4-6: duration
sampled_positive = df_fe["hab_class"] >= 1
sampled_bloom = df_fe["hab_class"] >= 2
df_fe["class_run_weeks"] = run_length(df_fe["last_class"]).clip(upper=52)
df_fe["weeks_since_pos"] = weeks_since(sampled_positive).clip(upper=104)
df_fe["weeks_since_bloom"] = weeks_since(sampled_bloom).clip(upper=104)
df_fe["pos_weeks_12wk"] = sampled_positive.astype(float).rolling(12, min_periods=1).sum()

# Feature 7: FWC category of the last observed max count
last_cells = (10 ** df_fe["last_log_cells"] - 1).round()
df_fe["last_fwc_level"] = pd.cut(
    last_cells,
    bins=[-np.inf, 1_000, 10_000, 100_000, 1_000_000, np.inf],
    labels=False
).astype(float)

# Verify the features

new_hab_features = [
    "last_class_prev",
    "class_change_1wk",
    "last_log_cells_chg1",
    "class_run_weeks",
    "weeks_since_pos",
    "weeks_since_bloom",
    "pos_weeks_12wk",
    "last_fwc_level"
]

print("HAB feature engineering completed!")
print("\nNew features created:")

for feature in new_hab_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Updated columns:", df_fe.shape[1])

print("\nOriginal rows:", df.shape[0])
print("Updated rows:", df_fe.shape[0])

# Confirm no observations were removed
assert len(df_fe) == len(df)

# The lag is the carried class one week earlier
assert (df_fe["last_class_prev"].iloc[1:].values == df_fe["last_class"].iloc[:-1].values).all()

# Missing values on labelled weeks: current-week class vs carried features
labelled = df_fe["y_class"].notna()
print("\nMissing on labelled weeks:")
print(df_fe.loc[labelled, ["hab_class"] + new_hab_features].isna().mean().map("{:.1%}".format))

# Preview results
display(
    df_fe[
        [
            "week_start",
            "hab_class",
            "last_class",
            "last_class_prev",
            "last_log_cells_chg1",
            "class_run_weeks",
            "weeks_since_bloom",
            "pos_weeks_12wk",
            "last_fwc_level"
        ]
    ].iloc[150:162]
)

HAB feature engineering completed!

New features created:
- last_class_prev
- class_change_1wk
- last_log_cells_chg1
- class_run_weeks
- weeks_since_pos
- weeks_since_bloom
- pos_weeks_12wk
- last_fwc_level

Original columns: 101
Updated columns: 109

Original rows: 595
Updated rows: 595

Missing on labelled weeks:
hab_class              4.3%
last_class_prev        0.2%
class_change_1wk       0.2%
last_log_cells_chg1    0.2%
class_run_weeks        0.0%
weeks_since_pos        7.4%
weeks_since_bloom      8.2%
pos_weeks_12wk         0.0%
last_fwc_level         0.0%
dtype: str


,week_start,hab_class,last_class,last_class_prev,last_log_cells_chg1,class_run_weeks,weeks_since_bloom,pos_weeks_12wk,last_fwc_level
150,2017-11-20,2.0,2.0,1.0,1.740144,1.0,0.0,6.0,4.0
151,2017-11-27,2.0,2.0,2.0,-0.620462,2.0,0.0,7.0,3.0
152,2017-12-04,2.0,2.0,2.0,-0.804155,3.0,0.0,8.0,3.0
153,2017-12-11,2.0,2.0,2.0,0.682305,4.0,0.0,9.0,3.0
154,2017-12-18,2.0,2.0,2.0,0.025468,5.0,0.0,10.0,3.0
155,2017-12-25,2.0,2.0,2.0,1.055857,6.0,0.0,10.0,4.0
156,2018-01-01,2.0,2.0,2.0,-1.050382,7.0,0.0,10.0,3.0
157,2018-01-08,2.0,2.0,2.0,0.355299,8.0,0.0,11.0,4.0
158,2018-01-15,2.0,2.0,2.0,-0.872047,9.0,0.0,12.0,3.0
159,2018-01-22,1.0,1.0,2.0,-0.482658,1.0,1.0,12.0,2.0



## 2. Seasonal Feature Engineering

This section creates cyclical seasonal features to represent annual patterns that may influence Harmful Algal Bloom (HAB) severity.

Our EDA showed that HAB severity varies across different periods of the year. Seasonal information may help models capture recurring environmental conditions associated with bloom development.

To represent seasonality continuously, we create two features:

1. **season_sin:** Sine transformation of the day of year.
2. **season_cos:** Cosine transformation of the day of year.

And two easy-to-read companions:

3. **month:** calendar month of the week.
4. **season_week:** weeks since 1 August. The bloom season runs from late summer into the next year, so this counts through a season without jumping at New Year.

### Benefits

- Capture recurring annual patterns in HAB severity.
- Represent the transition between December and January continuously.
- Provide seasonal information for linear and tree-based machine-learning models.

In [6]:
# Seasonal Feature Engineering

# Extract the day of year from the weekly date
day_of_year = df_fe["week_start"].dt.dayofyear

# Feature 8: Seasonal sine encoding
df_fe["season_sin"] = np.sin(
    2 * np.pi * (day_of_year - 1) / 365.25
)

# Feature 9: Seasonal cosine encoding
df_fe["season_cos"] = np.cos(
    2 * np.pi * (day_of_year - 1) / 365.25
)

# Feature 10: Month
df_fe["month"] = df_fe["week_start"].dt.month

# Feature 11: Weeks since 1 August of the bloom season holding the week
season_year = (df_fe["week_start"] - pd.DateOffset(months=7)).dt.year
season_start = pd.to_datetime(season_year.astype(str) + "-08-01")
df_fe["season_week"] = ((df_fe["week_start"] - season_start).dt.days // 7).astype(float)

new_seasonal_features = [
    "season_sin",
    "season_cos",
    "month",
    "season_week"
]

print("Seasonal feature engineering completed!")

print("\nNew features created:")
for feature in new_seasonal_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Updated columns:", df_fe.shape[1])

print("\nOriginal rows:", df.shape[0])
print("Updated rows:", df_fe.shape[0])

# Verify no observations were removed
assert len(df_fe) == len(df)

# Verify both cyclical features stay between -1 and 1, and season_week between 0 and 52
assert df_fe[["season_sin", "season_cos"]].abs().le(
    1.000001
).all().all()
assert df_fe["season_week"].between(0, 52).all()

# Preview seasonal features
display(
    df_fe[
        [
            "week_start",
            "hab_class",
            "season_sin",
            "season_cos",
            "month",
            "season_week"
        ]
    ].head(12)
)

Seasonal feature engineering completed!

New features created:
- season_sin
- season_cos
- month
- season_week

Original columns: 101
Updated columns: 113

Original rows: 595
Updated rows: 595


,week_start,hab_class,season_sin,season_cos,month,season_week
0,2015-01-05,0.0,0.068755,0.997634,1,22.0
1,2015-01-12,0.0,0.188099,0.982150,1,23.0
2,2015-01-19,0.0,0.304719,0.952442,1,24.0
3,2015-01-26,0.0,0.416926,0.908940,1,25.0
4,2015-02-02,0.0,0.523094,0.852275,2,26.0
5,2015-02-09,0.0,0.621687,0.783266,2,27.0
6,2015-02-16,0.0,0.711276,0.702913,2,28.0
7,2015-02-23,0.0,0.790563,0.612380,2,29.0
8,2015-03-02,0.0,0.858402,0.512978,3,30.0
9,2015-03-09,0.0,0.913808,0.406147,3,31.0



## 3. Environmental Feature Engineering

This section creates environmental features that summarize recent conditions over multiple weeks.

### Features to Create

**1. air_tmean_4wk**

The average air temperature over the current week and previous three weeks (at least three of them with data).

This feature captures sustained temperature conditions and reduces the influence of short-term fluctuations. It also
tracks Mote water temperature closely (checked below), and unlike Mote it covers almost every week.

**2. air_tmean_anom**

This week's air temperature minus its usual value for the same week of the year (2015–2021 reference, so later years
do not leak into it): warm or cold *for the time of year*.

**3. air_tmean_chg_1wk**

Week-on-week change in air temperature (cold fronts).

**4. uwnd_box_8wk**

The average east-west wind over the last eight weeks (at least four with data).

The 4-week chlorophyll anomaly (`chl_focus_0_40_anom_4wk`) and the onshore / offshore wind split
(`uwnd_westerly_mean`, `uwnd_easterly_mean`) are already made in notebook 03.

In [7]:
# Environmental Feature Engineering

# Feature 12: Four-week average air temperature
df_fe["air_tmean_4wk"] = df_fe["air_tmean_c"].rolling(
    window=4,
    min_periods=3
).mean()

# Feature 13: Air temperature anomaly vs the 2015-2021 mean of the same week of the year
week_of_year = df_fe["week_start"].dt.isocalendar().week.astype(int).clip(upper=52)
reference = df_fe["week_start"].dt.year <= 2021
weekly_mean = df_fe[reference].groupby(week_of_year[reference])["air_tmean_c"].mean().reindex(range(1, 53))
# Smooth over +/- 2 weeks, wrapping around New Year
padded = pd.concat([weekly_mean.iloc[-2:], weekly_mean, weekly_mean.iloc[:2]])
smooth = padded.rolling(5, center=True, min_periods=1).mean().iloc[2:-2]
smooth.index = range(1, 53)
df_fe["air_tmean_anom"] = df_fe["air_tmean_c"] - week_of_year.map(smooth).values

# Feature 14: Week-on-week air temperature change
df_fe["air_tmean_chg_1wk"] = df_fe["air_tmean_c"].diff()

# Feature 15: Eight-week average east-west wind
df_fe["uwnd_box_8wk"] = df_fe["uwnd_box"].rolling(
    window=8,
    min_periods=4
).mean()

# Verify engineered features

new_environmental_features = [
    "air_tmean_4wk",
    "air_tmean_anom",
    "air_tmean_chg_1wk",
    "uwnd_box_8wk"
]

print("Environmental feature engineering completed!")

print("\nNew features created:")
for feature in new_environmental_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Updated columns:", df_fe.shape[1])

print("\nOriginal rows:", df.shape[0])
print("Updated rows:", df_fe.shape[0])

# Verify no rows were removed
assert len(df_fe) == len(df)

# Air temperature as a stand-in for Mote water temperature
has_mote = df_fe["mote_temp_mean_c"].notna()
print(
    "\nWeeks with Mote:", int(has_mote.sum()), "of", len(df_fe),
    "| r(Mote, air_tmean_4wk) =",
    round(df_fe.loc[has_mote, "mote_temp_mean_c"].corr(df_fe.loc[has_mote, "air_tmean_4wk"]), 3),
    "| air_tmean_4wk available on", f"{df_fe['air_tmean_4wk'].notna().mean():.0%}", "of weeks"
)

# Preview original and engineered features
display(
    df_fe[
        [
            "week_start",
            "air_tmean_c",
            "air_tmean_4wk",
            "air_tmean_anom",
            "air_tmean_chg_1wk",
            "uwnd_box",
            "uwnd_box_8wk"
        ]
    ].iloc[208:220]
)

Environmental feature engineering completed!

New features created:
- air_tmean_4wk
- air_tmean_anom
- air_tmean_chg_1wk
- uwnd_box_8wk

Original columns: 101
Updated columns: 117

Original rows: 595
Updated rows: 595

Weeks with Mote: 191 of 595 | r(Mote, air_tmean_4wk) = 0.973 | air_tmean_4wk available on 96% of weeks


,week_start,air_tmean_c,air_tmean_4wk,air_tmean_anom,air_tmean_chg_1wk,uwnd_box,uwnd_box_8wk
208,2018-12-31,20.793651,18.244048,2.943311,0.714286,-0.011994,NaN
209,2019-01-07,17.142857,18.224206,-0.191610,-3.650794,-1.086529,NaN
210,2019-01-14,13.968254,17.996032,-2.463341,-3.174603,0.806530,NaN
211,2019-01-21,14.761905,16.666667,-2.158352,0.793651,-2.840819,-0.783203
212,2019-01-28,15.476190,15.337302,-1.955404,0.714286,-2.048579,-1.036278
213,2019-02-04,19.629630,15.958995,1.398715,4.153439,-1.697812,-1.146534
214,2019-02-11,19.523810,17.347884,0.612623,-0.105820,-0.885972,-1.109311
215,2019-02-18,23.452381,19.520503,3.647770,3.928571,-0.953116,-1.089786
216,2019-02-25,21.984127,21.147487,1.738473,-1.468254,-1.457795,-1.270511
217,2019-03-04,18.571429,20.882937,-2.058579,-3.412698,-0.768506,-1.230758



## 4. Add Regional Features to the Site-Week Table

The site-week table already holds each site's own history, its neighbours and the regional persistence of the same
week. Here we copy the new regional features onto it by `week_start`.

In [8]:
# Regional features to copy onto every site row of the same week
site_regional_features = [
    "season_sin",
    "season_cos",
    "month",
    "season_week",
    "weeks_since_bloom",
    "pos_weeks_12wk",
    "class_run_weeks",
    "air_tmean_4wk",
    "air_tmean_anom"
]

site_fe = site_df.merge(
    df_fe[["week_start"] + site_regional_features],
    on="week_start",
    how="left"
)

# Verify no site rows were added or removed
assert len(site_fe) == len(site_df)
assert not site_fe.duplicated(subset=["site_key", "week_start"]).any()

print("Site-week columns:", site_df.shape[1], "->", site_fe.shape[1])
display(site_fe[["site_key", "week_start", "site_last_class", "nbr_class", "reg_last_class"] + site_regional_features].head())

Site-week columns: 37 -> 46


,site_key,week_start,site_last_class,nbr_class,reg_last_class,season_sin,season_cos,month,season_week,weeks_since_bloom,pos_weeks_12wk,class_run_weeks,air_tmean_4wk,air_tmean_anom
0,26.71_-82.26,2015-03-09,NaN,0.0,0.0,0.913808,0.406147,3,31.0,NaN,0.0,10.0,20.347222,3.962963
1,26.71_-82.26,2015-03-16,0.0,0.0,0.0,0.955979,0.293434,3,32.0,NaN,0.0,11.0,22.648810,2.462963
2,26.71_-82.26,2015-10-12,0.0,0.0,2.0,-0.985056,0.172236,10,10.0,0.0,4.0,1.0,26.468254,-0.323507
3,26.71_-82.26,2015-10-19,1.0,2.0,2.0,-0.957232,0.289320,10,11.0,0.0,5.0,2.0,25.694444,-0.061602
4,26.71_-82.26,2015-11-02,1.0,0.0,2.0,-0.860600,0.509282,11,13.0,0.0,7.0,4.0,25.704365,4.623205



## 5. Highly Redundant Feature Analysis

This section identifies highly correlated numerical predictors that may provide overlapping information.

Removing unnecessary redundancy can:

- Simplify the feature set.
- Reduce multicollinearity in linear models.
- Improve model interpretability.
- Reduce unnecessary model complexity.
- Make feature selection more manageable for our small training dataset.

### Method

We calculate pairwise Pearson correlations between numerical predictor features using only the training dataset.

A correlation threshold of |r| >= 0.95 is used to flag highly redundant feature pairs for further review.

In [9]:
# Highly Redundant Feature Analysis

# Select training observations only
train_fe = df_fe[
    df_fe["split"] == "train"
].copy()

print("Training rows:", len(train_fe))
print("Training period:",
      train_fe["week_start"].min(),
      "to",
      train_fe["week_start"].max())

# Exclude non-predictor columns (keys, labels and availability flags)

exclude_columns = [
    col for col in df_fe.columns
    if col.startswith("y_")
    or col in ["target_year", "bloom_season", "label_effort_ok", "has_ksrq", "has_mote"]
    or col.endswith("_n_stations")
]

# Select numerical predictor candidates
numeric_features = [
    col for col in train_fe.select_dtypes(
        include="number"
    ).columns
    if col not in exclude_columns
]

print("\nNumber of numerical predictors examined:",
      len(numeric_features))


# Find highly correlated pairs

correlation_threshold = 0.95
minimum_overlap = 30

high_corr_pairs = []

for i, feature_1 in enumerate(numeric_features):

    for feature_2 in numeric_features[i + 1:]:

        # Identify observations where both values exist
        valid = train_fe[
            [feature_1, feature_2]
        ].notna().all(axis=1)

        n_overlap = valid.sum()

        # Require enough paired observations
        if n_overlap < minimum_overlap:
            continue

        x = train_fe.loc[valid, feature_1]
        y = train_fe.loc[valid, feature_2]

        # Skip constant features
        if x.nunique() < 2 or y.nunique() < 2:
            continue

        correlation = x.corr(y)

        if abs(correlation) >= correlation_threshold:

            high_corr_pairs.append({
                "Feature 1": feature_1,
                "Feature 2": feature_2,
                "Correlation": round(correlation, 4),
                "Absolute Correlation": round(
                    abs(correlation), 4
                ),
                "Overlapping Observations": int(n_overlap)
            })

# Display results

high_corr_df = pd.DataFrame(
    high_corr_pairs,
    columns=[
        "Feature 1",
        "Feature 2",
        "Correlation",
        "Absolute Correlation",
        "Overlapping Observations"
    ]
)

if not high_corr_df.empty:

    high_corr_df = high_corr_df.sort_values(
        "Absolute Correlation",
        ascending=False
    ).reset_index(drop=True)

    print("\nHIGHLY CORRELATED FEATURE PAIRS")
    print("-" * 50)

    display(high_corr_df)

else:
    print("\nNo feature pairs exceeded the threshold.")

print("\nCorrelation threshold:",
      correlation_threshold)

print("Minimum overlapping observations:",
      minimum_overlap)

Training rows: 468
Training period: 2015-01-05 00:00:00 to 2023-12-18 00:00:00

Number of numerical predictors examined: 98



HIGHLY CORRELATED FEATURE PAIRS
--------------------------------------------------


,Feature 1,Feature 2,Correlation,Absolute Correlation,Overlapping Observations
0,hab_class,last_class,1.0000,1.0000,365
1,nutr_bay_tkn_mgL_anom,nutr_bay_tn_mgL_anom,0.9993,0.9993,468
2,uwnd_box,uwnd_offshore,0.9928,0.9928,260
3,air_tmean_c,ksrq_air_temp_c,0.9923,0.9923,76
4,mote_temp_mean_c,mote_temp_max_c,0.9911,0.9911,137
5,n_habsos_samples,n_habsos_sites,0.9833,0.9833,365
6,mote_temp_mean_c,air_tmean_4wk,0.9711,0.9711,137
7,air_tmean_c,ksrq_dew_point_c,0.9701,0.9701,76
8,mote_temp_max_c,air_tmean_4wk,0.9688,0.9688,137
9,hab_class,last_fwc_level,0.9631,0.9631,365



Correlation threshold: 0.95
Minimum overlapping observations: 30


### Final Redundant Feature Removal

Remove selected redundant predictors from the reduced regional table. Each flagged pair (|r| >= 0.95 on training
rows) keeps the column that is better covered or already in the feature lists.

#### Features Selected for Removal

| Feature Removed | Feature Retained | Justification |
|---|---|---|
| hab_class | last_class | r = 1.00 on sampled weeks; last_class is also defined in unsampled weeks |
| last_fwc_level | last_class, last_log_cells | r = 0.96; the count itself is kept as last_log_cells |
| n_habsos_sites | n_habsos_samples | r = 0.98 |
| nutr_bay_tkn_mgL_anom | nutr_bay_tn_mgL_anom | r = 1.00 (TKN is most of TN in the Bay) |
| uwnd_offshore | uwnd_box | r = 0.99; the box mean has the longer validation against KSRQ |
| ksrq_air_temp_c, ksrq_dew_point_c, ksrq_wind_u | air_tmean_c, uwnd_box | r = 0.95-0.99; KSRQ starts Jul 2022 |
| mote_temp_max_c, mote_temp_mean_c | air_tmean_4wk | r = 0.97-0.99; Mote covers about a third of the weeks |
| air_tmin_c, air_tmax_c | air_tmean_c (+ air_trange_c) | r = 0.95-0.96; the daily range keeps what the mean does not |
| chl_focus_0_40_med | chl_focus_10_40_med | r = 0.96; the offshore band is the candidate feature |

In [10]:
#  Final Redundant Feature Removal

# Remove only the selected features
features_to_remove = [
    "hab_class",
    "last_fwc_level",
    "n_habsos_sites",
    "nutr_bay_tkn_mgL_anom",
    "uwnd_offshore",
    "ksrq_air_temp_c",
    "ksrq_dew_point_c",
    "ksrq_wind_u",
    "mote_temp_max_c",
    "mote_temp_mean_c",
    "air_tmin_c",
    "air_tmax_c",
    "chl_focus_0_40_med"
]

# Create reduced dataset without modifying df_fe
df_fe_reduced = df_fe.drop(
    columns=features_to_remove
).copy()

# Verify the carried HAB features and the temperature range are retained
assert "last_log_cells" in df_fe_reduced.columns
assert "last_class_prev" in df_fe_reduced.columns
assert "air_trange_c" in df_fe_reduced.columns

# Verify dataset dimensions
assert df_fe_reduced.shape == (len(df), df_fe.shape[1] - len(features_to_remove))

print("REDUNDANT FEATURE REMOVAL COMPLETED")
print("-" * 50)

print("\nFeatures removed:")
for feature in features_to_remove:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Full engineered columns:", df_fe.shape[1])
print("Reduced columns:", df_fe_reduced.shape[1])

print("\nOriginal rows:", len(df))
print("Reduced rows:", len(df_fe_reduced))

REDUNDANT FEATURE REMOVAL COMPLETED
--------------------------------------------------

Features removed:
- hab_class
- last_fwc_level
- n_habsos_sites
- nutr_bay_tkn_mgL_anom
- uwnd_offshore
- ksrq_air_temp_c
- ksrq_dew_point_c
- ksrq_wind_u
- mote_temp_max_c
- mote_temp_mean_c
- air_tmin_c
- air_tmax_c
- chl_focus_0_40_med

Original columns: 101
Full engineered columns: 117
Reduced columns: 104

Original rows: 595
Reduced rows: 595



## 6. Feature Sets for Modelling

We tested which feature groups improve next-week forecasts on years the model has not seen. Each test year was
predicted by a model trained on earlier years only, with test weeks assigned by `target_year`. Models were a
regularised logistic regression and a small LightGBM. "Onset" weeks are weeks whose last observed class was 0, where
persistence alone cannot help.

Logistic regression results (the small LightGBM scored lower on the 595-week regional table):

| Table | Feature set | AUC (≥ 5k next week) | Average precision | Onset AUC |
|---|---|---|---|---|
| Regional (201 test weeks, 10 onsets) | Persistence rule (next week = last class) | 0.880 | 0.737 | 0.500 |
| | Notebook-05 features as first written (current-week HAB lags + environment) | 0.895 | 0.828 | 0.681 |
| | HAB history + every environmental group | 0.801 | 0.632 | 0.563 |
| | **`regional_core`** (carried HAB history + duration + season) | **0.929** | **0.869** | **0.754** |
| | core + temperature | 0.934 | 0.872 | 0.781 |
| | core + chlorophyll | 0.903 | 0.812 | 0.668 |
| | core + wind | 0.862 | 0.796 | 0.579 |
| | core + Bay nutrients | 0.905 | 0.829 | 0.743 |
| | core + rain | 0.899 | 0.821 | 0.662 |
| Site-week (6,348 test rows, 248 onsets) | Site / neighbour / regional history + environment | 0.907 | 0.438 | 0.890 |
| | **`site_core`** | **0.926** | **0.506** | **0.909** |
| | core + temperature | 0.930 | 0.504 | 0.914 |
| | core + chlorophyll | 0.923 | 0.499 | 0.904 |
| | core + wind | 0.926 | 0.496 | 0.916 |
| | core + rain | 0.923 | 0.488 | 0.906 |

The regional test has only 10 onset weeks, so its onset AUC moves a lot with one or two weeks; treat differences
below ~0.05 there as noise.

Findings:
- **HAB history dominates**, and how long the current state has lasted adds to it.
- Environmental groups (chlorophyll, wind, rain, nutrients) **did not improve** these out-of-year forecasts with
  simple models; most made them slightly worse. Temperature is the only group with a small positive effect at both grains.
- So the feature lists below have a **core** set and environmental **candidates**. Add a candidate group to the core and
  keep it only if it helps on the validation year.
- Bradenton rain is an **optional** candidate. It has not helped in any test, and its record ends 31 Dec 2025, so the
  2026 holdout weeks have no rain values.

In [11]:
# Feature sets for modelling

feature_sets = {
    "labels": {
        "primary": "y_class",
        "binary": "y_binary",
        "alternatives": ["y_class_10k", "y_class_box", "y_class_mote_25km"]
    },
    "split_column": "split",
    "regional_core": [
        "last_class", "last_log_cells", "last_class_prev", "last_log_cells_chg1", "weeks_since_obs",
        "class_max_4wk", "n_habsos_samples", "weeks_since_bloom", "pos_weeks_12wk", "class_run_weeks",
        "season_sin", "season_cos"
    ],
    "regional_candidates": {
        "temperature": ["air_tmean_4wk", "air_tmean_anom", "air_trange_c"],
        "chlorophyll": ["chl_focus_0_40_anom", "chl_focus_0_40_anom_4wk", "chl_focus_10_40_med"],
        "wind": ["uwnd_box_4wk", "uwnd_westerly_mean", "uwnd_easterly_mean"],
        "nutrients_bay": ["nutr_bay_tn_mgL_anom", "nutr_bay_tp_mgL_anom", "nutr_bay_nox_mgL_anom",
                          "nutr_bay_chla_corr_ugL_anom"],
        "rain_optional": ["rain_total_mm", "rain_days", "rain_4wk_mm", "rain_8wk_mm"]
    },
    "site_core": [
        "site_last_class", "site_last_log_cells", "site_weeks_since_obs", "nbr_class", "reg_last_class",
        "reg_class_max_4wk", "reg_n_samples", "weeks_since_bloom", "pos_weeks_12wk", "class_run_weeks",
        "season_sin", "season_cos"
    ],
    "site_candidates": {
        "temperature": ["air_tmean_4wk", "air_tmean_anom"],
        "chlorophyll": ["site_log_chl", "site_log_chl_anom", "site_log_chl_anom_4wk"],
        "wind": ["uwnd_box_4wk", "uwnd_westerly_mean", "uwnd_easterly_mean"],
        "rain_optional": ["rain_4wk_mm", "rain_8wk_mm"]
    },
    "do_not_use": [
        "hab_class (current week, missing when unsampled; use last_class)",
        "site_class_now / site_log_cells_now (current week; use site_last_class)",
        "ksrq_* (starts Jul 2022, no training data)",
        "mote_* (about a third of weeks; air_tmean_4wk stands in)",
        "y_*, target_week, target_year, bloom_season, split, label_effort_ok (labels and keys)"
    ]
}

# Every listed feature must exist in its table
regional_listed = feature_sets["regional_core"] + sum(feature_sets["regional_candidates"].values(), [])
site_listed = feature_sets["site_core"] + sum(feature_sets["site_candidates"].values(), [])
assert set(regional_listed) <= set(df_fe_reduced.columns), set(regional_listed) - set(df_fe_reduced.columns)
assert set(site_listed) <= set(site_fe.columns), set(site_listed) - set(site_fe.columns)

# Coverage of the core features on labelled rows
print("Missing values on labelled regional rows (core features):")
print(df_fe.loc[df_fe["y_class"].notna(), feature_sets["regional_core"]].isna().mean().map("{:.1%}".format))

Missing values on labelled regional rows (core features):
last_class             0.0%
last_log_cells         0.0%
last_class_prev        0.2%
last_log_cells_chg1    0.2%
weeks_since_obs        0.0%
class_max_4wk          1.4%
n_habsos_samples       0.0%
weeks_since_bloom      8.2%
pos_weeks_12wk         0.0%
class_run_weeks        0.0%
season_sin             0.0%
season_cos             0.0%
dtype: str



## Export Feature-Engineered Datasets

### Feature Engineering Summary

- Regional table: new HAB history / duration, seasonal and environmental features.
- Site-week table: the regional features of the same week.
- Redundant features removed from the reduced regional table.
- Observations preserved: every row of both tables.

### Output Files

**model_table_weekly_fe_full.parquet**: complete feature-engineered regional table.

**model_table_weekly_fe_reduced.parquet**: the same without the redundant features.

**site_week_table_fe.parquet**: feature-engineered site-week table.

**feature_sets.json**: core and candidate feature lists, labels and split column.

### Data Integrity and Handoff

- All rows are preserved.
- Missing values remain unchanged (no imputation; fit any imputer on training rows only).
- The notebook-03 tables are not overwritten.

In [12]:
# Export Feature-Engineered Datasets

output_dir = Path(PROCESSED)

full_output_path = output_dir / "model_table_weekly_fe_full.parquet"
reduced_output_path = output_dir / "model_table_weekly_fe_reduced.parquet"
site_output_path = output_dir / "site_week_table_fe.parquet"
feature_sets_path = output_dir / "feature_sets.json"

# Make sure we are not overwriting the original files
assert full_output_path != Path(data_path)
assert site_output_path != Path(site_path)

# -----------------------------------
# Save the datasets
# -----------------------------------

df_fe.to_parquet(full_output_path, index=False)
df_fe_reduced.to_parquet(reduced_output_path, index=False)
site_fe.to_parquet(site_output_path, index=False)
feature_sets_path.write_text(json.dumps(feature_sets, indent=2))

# Reload files to verify the exports

full_check = pd.read_parquet(full_output_path)
reduced_check = pd.read_parquet(reduced_output_path)
site_check = pd.read_parquet(site_output_path)

pd.testing.assert_frame_equal(full_check, df_fe)
pd.testing.assert_frame_equal(reduced_check, df_fe_reduced)
pd.testing.assert_frame_equal(site_check, site_fe)
assert json.loads(feature_sets_path.read_text()) == feature_sets


# Display export results

print("DATASETS EXPORTED SUCCESSFULLY!")
print("-" * 50)

for path, frame in [(full_output_path, full_check), (reduced_output_path, reduced_check), (site_output_path, site_check)]:
    print(f"{path.name:40s} shape {frame.shape}")
print(f"{feature_sets_path.name:40s} {len(regional_listed)} regional / {len(site_listed)} site features listed")

print("\nAll verification checks passed!")
print("Notebook-03 tables were not overwritten.")
print("All rows were preserved.")

DATASETS EXPORTED SUCCESSFULLY!
--------------------------------------------------
model_table_weekly_fe_full.parquet       shape (595, 117)
model_table_weekly_fe_reduced.parquet    shape (595, 104)
site_week_table_fe.parquet               shape (21718, 46)
feature_sets.json                        29 regional / 22 site features listed

All verification checks passed!
Notebook-03 tables were not overwritten.
All rows were preserved.
